# Find the Closest Forward Model to the Experimental Data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import euclidean


def count_close_points(exp_resistances_filtered, other_interpolated, match_threshold=0.05):
    """
    count how many points are near-perfectly matched (log10 residual below a threshold),
    rather than summarizing total error -- a candidate with many closely-matched points
    is preferred over one with a lower aggregate error driven by just a few very good points

    parameters:
    - exp_resistances_filtered: experimental resistance values (kohm)
    - other_interpolated: candidate resistance values, interpolated onto the same distances (kohm)
    - match_threshold: max |log10(other) - log10(exp)| for a point to count as "matched"
      (default 0.05, i.e. about 12% relative error)

    returns:
    - n_close: number of points within match_threshold
    - residual: full per-point log10 residual array (for tie-breaking / reporting)
    """
    residual = np.log10(other_interpolated) - np.log10(exp_resistances_filtered)
    n_close = int(np.sum(np.abs(residual) < match_threshold))
    return n_close, residual


def find_closest_traces(csv_file, list_of_distances, list_of_average_traces,
                   labels, distance_range=None, tolerance=0.05, match_threshold=0.05):
    """
    find modeled traces with the most closely-matched points to experimental average traces

    parameters:
    - csv_file: path to csv file with forward modeled traces
    - list_of_distances: list of distance arrays in um, one per trace
    - list_of_average_traces: list of resistance arrays in kohm, one per trace
    - labels: list of trace names
    - distance_range: tuple (min_um, max_um) to filter distances, or None for all
    - tolerance: relative tolerance (on the tie-break magnitude) for considering matches
      equally good among candidates tied on n_close_points (default 0.05)
    - match_threshold: max |log10 residual| for a point to count as "matched" (default 0.05,
      i.e. about 12% relative error)

    returns:
    - dict with {trace_name: {'resistivity_profiles': list, 'n_close_points': int, ...}}
    """
    # load the csv file
    df = pd.read_csv(csv_file)

    # extract resistivity column names (first 5 columns)
    resistivity_cols = df.columns[:5].tolist()

    # extract distance columns (6th column onward) and convert to numeric
    distance_um = df.columns[5:].tolist()
    distances_um = np.array([float(distance) for distance in distance_um])

    # convert modeled resistances from Mohm to kohm (multiply by 1000)
    modeled_resistances = df[distance_um].values * 1000

    results = {}

    for trace_name, exp_distances, exp_resistances in zip(labels, list_of_distances,
                                                           list_of_average_traces):
        # filter experimental data to match distance range if specified
        if distance_range is not None:
            mask = (exp_distances >= distance_range[0]) & (exp_distances <= distance_range[1])
            exp_distances_filtered = exp_distances[mask]
            exp_resistances_filtered = exp_resistances[mask]
        else:
            exp_distances_filtered = exp_distances
            exp_resistances_filtered = exp_resistances

        # interpolate modeled data to match experimental distances
        modeled_interpolated = np.array([
            np.interp(exp_distances_filtered, distances_um, modeled_row)
            for modeled_row in modeled_resistances
        ])

        # count how many points are near-perfectly matched for every candidate row
        match_counts = np.zeros(len(modeled_interpolated), dtype=int)
        residuals_all = np.zeros_like(modeled_interpolated)
        for i, modeled_row in enumerate(modeled_interpolated):
            n_close, residual = count_close_points(exp_resistances_filtered, modeled_row, match_threshold)
            match_counts[i] = n_close
            residuals_all[i] = residual

        # magnitude used only to break ties among candidates with the same n_close_points
        magnitudes = np.sqrt(np.sum(residuals_all ** 2, axis=1))

        # select candidates with the most matched points, then narrow by magnitude
        best_count = np.max(match_counts)
        tied_indices = np.where(match_counts == best_count)[0]
        min_magnitude = np.min(magnitudes[tied_indices])
        closest_indices = tied_indices[magnitudes[tied_indices] <= min_magnitude * (1 + tolerance)]

        # extract resistivity profiles for all equally good matches
        resistivity_profiles = [df.iloc[idx][resistivity_cols].to_dict()
                               for idx in closest_indices]

        # store modeled traces for plotting, limited to the compared (filtered) distances
        best_match_traces = [modeled_interpolated[idx] for idx in closest_indices]

        results[trace_name] = {
            'resistivity_profiles': resistivity_profiles,
            'n_close_points': int(best_count),
            'total_points': len(exp_resistances_filtered),
            'tie_break_distance': float(min_magnitude),
            'model_indices': closest_indices.tolist(),
            'best_match_distances': exp_distances_filtered,
            'best_match_resistances': best_match_traces
        }

    return results


def plot_trace_comparison(trace_name, exp_distances, exp_resistances, exp_std,
                          best_match_distances, best_match_resistances_list,
                          resistivity_profiles, save_path=None):
    """
    plot experimental trace with std band and best matching modeled trace(s)

    parameters:
    - trace_name: name of the trace for plot title
    - exp_distances: experimental distance array in um
    - exp_resistances: experimental resistance array in kohm
    - exp_std: standard deviation array in kohm
    - best_match_distances: modeled trace distances in um
    - best_match_resistances_list: list of modeled trace resistances in kohm
    - resistivity_profiles: list of dicts of resistivity values for the 5 layers
    - save_path: optional path to save figure
    """
    fig = plt.figure(figsize=(12, 6))

    # create gridspec with two columns: plot (wider) and text (narrower)
    gs = fig.add_gridspec(1, 2, width_ratios=[4, 1], wspace=0.01)
    ax = fig.add_subplot(gs[0])
    ax_text = fig.add_subplot(gs[1])
    ax_text.axis('off')

    # plot experimental trace with std band
    ax.plot(exp_distances, exp_resistances, 'o-', color='#2E86AB',
            linewidth=2, markersize=6, label='Experimental', zorder=3)
    ax.fill_between(exp_distances,
                     exp_resistances - exp_std,
                     exp_resistances + exp_std,
                     color='#2E86AB', alpha=0.2)

    # plot all best match modeled traces
    colors = ['#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4B51']
    for i, best_match_resistances in enumerate(best_match_resistances_list):
        color = colors[i % len(colors)]
        label = f'Closest Forward Model {i+1}' if len(best_match_resistances_list) > 1 else 'Closest Forward Model'
        ax.plot(best_match_distances, best_match_resistances, '--',
                color=color, linewidth=2, label=label, zorder=2)

    # format resistivity profile text for all matches
    profile_text = f'Resistivity profile(s) (Ω·cm)\n{len(resistivity_profiles)} match(es):\n\n'
    for i, resistivity_profile in enumerate(resistivity_profiles):
        if len(resistivity_profiles) > 1:
            profile_text += f'Match {i+1}:\n'
        for layer, value in resistivity_profile.items():
            profile_text += f'{layer}: {value:.1f}\n'
        if i < len(resistivity_profiles) - 1:
            profile_text += '\n'

    # add text box in the separate axis
    ax_text.text(0.05, 0.95, profile_text.strip(),
                transform=ax_text.transAxes,
                fontsize=9,
                verticalalignment='top',
                horizontalalignment='left',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # labels and formatting
    ax.set_xlabel('Distance (μm)', fontsize=12)
    ax.set_ylabel('Resistance (kΩ)', fontsize=12)
    ax.set_title(f'{trace_name}', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlim(exp_distances.min(), exp_distances.max())

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    return fig, ax


def plot_resistivity_boxplots(results, save_dir=None):
    """
    create horizontal box plots for resistivity values across all layers

    parameters:
    - results: dict from find_closest_traces containing all matches
    - save_dir: optional directory to save figures

    returns:
    - list of figure objects
    """
    # get layer names from first result (assumes all have same layers)
    first_result = next(iter(results.values()))
    layer_names = list(first_result['resistivity_profiles'][0].keys())

    # remove saline from names
    layer_names.remove('rho_saline')
    n_layers = len(layer_names)

    figures = []

    # plot individual trace boxplots
    for trace_name, result in results.items():
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))

        # collect resistivity values for each layer
        layer_data = {layer: [] for layer in layer_names}
        for profile in result['resistivity_profiles']:
            for layer, value in profile.items():
                # skip if saline
                if layer == 'rho_saline':
                    continue
                layer_data[layer].append(value)

        # prepare data for horizontal boxplot
        data_to_plot = [layer_data[layer] for layer in layer_names]
        positions = np.arange(n_layers, 0, -1)  # reverse order for top-to-bottom

        # create horizontal boxplot
        bp = ax.boxplot(data_to_plot, positions=positions, vert=False, patch_artist=True,
                       widths=0.6, showfliers=True,
                       boxprops=dict(facecolor='#A23B72', alpha=0.6),
                       medianprops=dict(color='black', linewidth=2),
                       whiskerprops=dict(color='#A23B72', linewidth=1.5),
                       capprops=dict(color='#A23B72', linewidth=1.5))

        # labels and formatting
        ax.set_yticks(positions)
        ax.set_yticklabels(layer_names)
        ax.set_xlabel('Resistivity (Ω·cm)', fontsize=11)
        ax.set_title(trace_name, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')
        ax.set_ylim(0.5, n_layers + 0.5)

        plt.tight_layout()

        if save_dir:
            save_path = f"{save_dir}/{trace_name}_resistivity_boxplot.png"
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        figures.append(fig)

    # plot combined boxplot for all traces
    fig_combined, ax_combined = plt.subplots(1, 1, figsize=(12, 8))

    # collect all resistivity values across all traces
    all_layer_data = {layer: [] for layer in layer_names}
    for result in results.values():
        for profile in result['resistivity_profiles']:
            for layer, value in profile.items():
                if layer == 'rho_saline':
                    continue
                all_layer_data[layer].append(value)

    # prepare data for horizontal boxplot
    all_data_to_plot = [all_layer_data[layer] for layer in layer_names]
    positions = np.arange(n_layers, 0, -1)

    # create horizontal boxplot
    bp = ax_combined.boxplot(all_data_to_plot, positions=positions, vert=False,
                             patch_artist=True, widths=0.6, showfliers=True,
                             boxprops=dict(facecolor='#2E86AB', alpha=0.6),
                             medianprops=dict(color='black', linewidth=2),
                             whiskerprops=dict(color='#2E86AB', linewidth=1.5),
                             capprops=dict(color='#2E86AB', linewidth=1.5))

    # labels and formatting
    ax_combined.set_yticks(positions)
    ax_combined.set_yticklabels(layer_names)
    ax_combined.set_xlabel('Resistivity (Ω·cm)', fontsize=11)
    ax_combined.set_title('All Traces Combined', fontsize=12, fontweight='bold')
    ax_combined.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax_combined.set_ylim(0.5, n_layers + 0.5)

    plt.tight_layout()

    if save_dir:
        save_path = f"{save_dir}/All_Traces_Combined_resistivity_boxplot.png"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    figures.append(fig_combined)

    return figures


def plot_all_comparisons(csv_file, list_of_distances, list_of_average_traces,
                        list_of_stds, labels, distance_range=None, tolerance=0.05,
                        match_threshold=0.05):
    """
    find best matches and plot all trace comparisons

    parameters:
    - csv_file: path to csv file with forward modeled traces
    - list_of_distances: list of distance arrays in um, one per trace
    - list_of_average_traces: list of resistance arrays in kohm, one per trace
    - list_of_stds: list of std arrays in kohm, one per trace
    - labels: list of trace names
    - distance_range: tuple (min_um, max_um) to filter distances, or None for all
    - tolerance: relative tolerance on the tie-break magnitude (default 0.05)
    - match_threshold: max |log10 residual| for a point to count as "matched" (default 0.05)

    returns:
    - results dict from find_closest_traces
    - list of figure objects
    """
    # find closest traces
    results = find_closest_traces(csv_file, list_of_distances,
                                  list_of_average_traces, labels, distance_range,
                                  tolerance, match_threshold)

    # create plots
    figures = []
    for trace_name, exp_distances, exp_resistances, exp_std in zip(labels,
                                                                     list_of_distances,
                                                                     list_of_average_traces,
                                                                     list_of_stds):
        result = results[trace_name]

        fig, ax = plot_trace_comparison(
            trace_name,
            exp_distances,
            exp_resistances,
            exp_std,
            result['best_match_distances'],
            result['best_match_resistances'],
            result['resistivity_profiles'],
        )
        figures.append(fig)

    # add box plot
    figs = plot_resistivity_boxplots(results)
    figures.extend(figs)


    return results, figures


def load_trace_from_xlsx(xlsx_path, spacing_um=33.4, exclude_first=True, unit_scale=1.0):
    """
    load a single averaged trace from an excel file with columns V_0..V_n

    parameters:
    - xlsx_path: path to excel file with a single row of V_0..V_n resistance values
    - spacing_um: spacing between successive samples in um (V_i is at spacing_um * i)
    - exclude_first: if True, drop the V_0 (0 um) point from the returned trace
    - unit_scale: multiplier applied to raw values to convert to kohm (e.g. 1000 for Mohm -> kohm)

    returns:
    - distances_um: array of distances in um
    - resistances_kohm: array of resistance values in kohm
    """
    df = pd.read_excel(xlsx_path)
    value_cols = [col for col in df.columns if str(col).startswith('V_')]
    resistances_kohm = df.iloc[0][value_cols].values.astype(float) * unit_scale
    distances_um = spacing_um * np.arange(len(value_cols))

    if exclude_first:
        distances_um = distances_um[1:]
        resistances_kohm = resistances_kohm[1:]

    return distances_um, resistances_kohm


def compute_match_count(exp_distances, exp_resistances, other_distances, other_resistances,
                         distance_range=None, match_threshold=0.05):
    """
    count how many points of another trace are near-perfectly matched to the experimental
    trace, interpolating the other trace onto the (optionally range-filtered) experimental
    distances -- mirrors the counting logic used in find_closest_traces so the ML trace and
    forward-model matches are scored the same way

    parameters:
    - exp_distances, exp_resistances: experimental trace (um, kohm)
    - other_distances, other_resistances: comparison trace (um, kohm)
    - distance_range: tuple (min_um, max_um) to filter the comparison window, or None for all
    - match_threshold: max |log10 residual| for a point to count as "matched" (default 0.05)

    returns:
    - n_close: number of matched points
    - total_points: number of points compared
    - other_interpolated: the comparison trace resampled onto the filtered experimental distances
    """
    if distance_range is not None:
        mask = (exp_distances >= distance_range[0]) & (exp_distances <= distance_range[1])
        exp_distances_filtered = exp_distances[mask]
        exp_resistances_filtered = exp_resistances[mask]
    else:
        exp_distances_filtered = exp_distances
        exp_resistances_filtered = exp_resistances

    other_interpolated = np.interp(exp_distances_filtered, other_distances, other_resistances)
    n_close, _ = count_close_points(exp_resistances_filtered, other_interpolated, match_threshold)

    return n_close, len(exp_resistances_filtered), other_interpolated


def plot_ml_vs_model_comparison(trace_name, exp_distances, exp_resistances, exp_std,
                                 best_match_distances, best_match_resistances_list,
                                 best_match_n_close, total_points, resistivity_profiles,
                                 decoded_distances, decoded_resistances,
                                 distance_range=None, match_threshold=0.05, save_path=None):
    """
    plot experimental trace together with the best forward-model match(es) and the
    ML-decoded trace, reporting how many points each one matches closely (rather than
    a single aggregate error number)

    parameters:
    - trace_name: name of the trace for plot title
    - exp_distances, exp_resistances, exp_std: experimental trace (um, kohm, kohm)
    - best_match_distances: distance array shared by the modeled trace(s) (um)
    - best_match_resistances_list: list of best-fit modeled resistance arrays (kohm)
    - best_match_n_close: number of closely-matched points for the forward-model match
    - total_points: total number of points compared
    - resistivity_profiles: list of dicts of resistivity values (Ohm*cm), one per best match
    - decoded_distances, decoded_resistances: ML-decoded trace (um, kohm)
    - distance_range: tuple (min_um, max_um) used to restrict the comparison window, or None
    - match_threshold: max |log10 residual| for a point to count as "matched" (default 0.05)
    - save_path: optional path to save figure

    returns:
    - fig, ax, ml_n_close
    """
    ml_n_close, _, _ = compute_match_count(
        exp_distances, exp_resistances, decoded_distances, decoded_resistances,
        distance_range, match_threshold
    )

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(exp_distances, exp_resistances, 'o-', color='#2E86AB',
            linewidth=2, markersize=6, label='Experimental', zorder=3)
    ax.fill_between(exp_distances, exp_resistances - exp_std, exp_resistances + exp_std,
                     color='#2E86AB', alpha=0.2)

    layer_names = [layer for layer in resistivity_profiles[0].keys() if layer != 'rho_saline']

    colors = ['#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4B51']
    for i, (best_match_resistances, profile) in enumerate(zip(best_match_resistances_list, resistivity_profiles)):
        color = colors[i % len(colors)]
        profile_str = ', '.join(f'{layer}={profile[layer]:.1f}' for layer in layer_names)
        prefix = f'Model match {i+1}' if len(best_match_resistances_list) > 1 else 'Model match'
        label = f'{prefix}: {profile_str}'
        ax.plot(best_match_distances, best_match_resistances, '--',
                color=color, linewidth=2, label=label, zorder=2)

    ax.plot(decoded_distances, decoded_resistances, '-', color='#3A0CA3',
            linewidth=2.5, label='ML Decoded', zorder=2)

    ax.set_xlabel('Distance (μm)', fontsize=12)
    ax.set_ylabel('Resistance (kΩ)', fontsize=12)
    ax.set_title(f'{trace_name}: ML Decoded vs Forward-Model Match', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_yscale('log')
    ax.set_xscale('log')
    # ax.set_xlim(exp_distances.min(), exp_distances.max())

    metrics_text = (f'Closely-matched points (|log10 residual| < {match_threshold}):\n'
                     f'Forward-model match: {best_match_n_close}/{total_points}\n'
                     f'ML decoded: {ml_n_close}/{total_points}')
    ax.text(0.02, 0.02, metrics_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom', horizontalalignment='left',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    return fig, ax, ml_n_close


def plot_residuals(trace_name, exp_distances, exp_resistances,
                    best_match_distances, best_match_resistances,
                    decoded_distances, decoded_resistances,
                    distance_range=None, match_threshold=0.05, save_path=None):
    """
    plot per-point log10 residuals (model - experimental, decoded - experimental) with the
    match_threshold band shaded, so you can see exactly which points count as "matched"
    for each trace, rather than only a summary count

    parameters:
    - trace_name: name of the trace for plot title
    - exp_distances, exp_resistances: experimental trace (um, kohm)
    - best_match_distances, best_match_resistances: best forward-model match (um, kohm)
    - decoded_distances, decoded_resistances: ML-decoded trace (um, kohm)
    - distance_range: tuple (min_um, max_um) to restrict the comparison window, or None
    - match_threshold: max |log10 residual| for a point to count as "matched" (default 0.05)
    - save_path: optional path to save figure

    returns:
    - fig, ax
    """
    if distance_range is not None:
        mask = (exp_distances >= distance_range[0]) & (exp_distances <= distance_range[1])
        exp_distances_filtered = exp_distances[mask]
        exp_resistances_filtered = exp_resistances[mask]
    else:
        exp_distances_filtered = exp_distances
        exp_resistances_filtered = exp_resistances

    log_exp = np.log10(exp_resistances_filtered)

    model_interp = np.interp(exp_distances_filtered, best_match_distances, best_match_resistances)
    decoded_interp = np.interp(exp_distances_filtered, decoded_distances, decoded_resistances)

    model_residual = np.log10(model_interp) - log_exp
    decoded_residual = np.log10(decoded_interp) - log_exp

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.axhline(0, color='gray', linewidth=1, linestyle=':')
    ax.axhspan(-match_threshold, match_threshold, color='green', alpha=0.1,
               label=f'"matched" band (±{match_threshold})')

    ax.plot(exp_distances_filtered, model_residual, 'o-', color='#A23B72', label='Forward-model match')
    ax.plot(exp_distances_filtered, decoded_residual, 'o-', color='#3A0CA3', label='ML decoded')

    ax.set_xlabel('Distance (μm)', fontsize=12)
    ax.set_ylabel('log10(model) - log10(experimental)', fontsize=12)
    ax.set_title(f'{trace_name}: per-point log10 residual', fontsize=13, fontweight='bold')
    ax.set_xscale('log')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    return fig, ax


# define path to experimental file
experimental_file_path = (r'.../average_trace_all_first_600um.xlsx')

decoded_file_path = (r'.../average_trace_all_first_600um_ML_decoded_4layers.xlsx')

match_threshold = 0.01  # |log10 residual| below this counts as a "matched" point (~12% error)

exp_distances, exp_resistances = load_trace_from_xlsx(experimental_file_path)

list_of_distances = [exp_distances]
list_of_average_traces = [exp_resistances]
list_of_stds = [np.zeros_like(exp_resistances)]  # no std/replicate data in this file
labels = ["Average Trace (first 600um)"]

# find closest matches and create plots
results, figures = plot_all_comparisons(
	'full_training_set_33um.csv',
    list_of_distances,
    list_of_average_traces,
    list_of_stds,
    labels,
    distance_range=(exp_distances.min(), exp_distances.max()),
    tolerance=0.01,
    match_threshold=match_threshold
)

# load the ML-decoded trace and compare it against experimental + best forward-model match
decoded_distances, decoded_resistances = load_trace_from_xlsx(decoded_file_path, unit_scale=1000)

trace_name = labels[0]
result = results[trace_name]

fig_ml, ax_ml, ml_n_close = plot_ml_vs_model_comparison(
    trace_name,
    exp_distances,
    exp_resistances,
    list_of_stds[0],
    result['best_match_distances'],
    result['best_match_resistances'],
    result['n_close_points'],
    result['total_points'],
    result['resistivity_profiles'],
    decoded_distances,
    decoded_resistances,
    distance_range=(exp_distances.min(), exp_distances.max()),
    match_threshold=match_threshold
)
figures.append(fig_ml)

fig_res, ax_res = plot_residuals(
    trace_name,
    exp_distances,
    exp_resistances,
    result['best_match_distances'],
    result['best_match_resistances'][0],
    decoded_distances,
    decoded_resistances,
    distance_range=(exp_distances.min(), exp_distances.max()),
    match_threshold=match_threshold
)
figures.append(fig_res)

plt.show()

# Find NFL+GCL that is equal to IPL (Synthetic Data Only)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree


def plot_rows_matching_all_but_first_resistivity(csv_file, rel_tolerance=0.01, unit_scale=1000):
    """
    find groups of rows in csv_file whose resistivity values (all columns except the
    first) agree within a relative tolerance, and plot each group's resistance vs.
    distance on a single log-log figure (one new figure per group)

    matching is done in log10 space with a Chebyshev (max) norm, so two rows are
    grouped only if EVERY matched column agrees within rel_tolerance -- not just on
    average. this dataset has no exact duplicates (it's a random/LHS sampling design,
    not a grid), so exact equality would never find a match.

    parameters:
    - csv_file: path to csv file with resistivity columns (first 5) followed by distance columns
    - rel_tolerance: max relative difference allowed on each matched resistivity column
      (default 0.01, i.e. 1%)
    - unit_scale: multiplier applied to raw resistance values to convert to kohm (default 1000, Mohm->kohm)

    returns:
    - list of (fig, ax) tuples, one per matching group
    """
    df = pd.read_csv(csv_file)

    resistivity_cols = df.columns[:5].tolist()
    distance_cols = df.columns[5:].tolist()
    distances_um = np.array([float(d) for d in distance_cols])

    first_col, match_cols = resistivity_cols[0], resistivity_cols[1:]
    log_values = np.log10(df[match_cols].values)
    radius = np.log10(1 + rel_tolerance)

    pairs = cKDTree(log_values).query_pairs(r=radius, p=np.inf)

    # union-find to merge overlapping pairs into groups (a group can exceed 2 rows)
    parent = list(range(len(df)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj

    groups = {}
    for i in range(len(df)):
        groups.setdefault(find(i), []).append(i)

    figures = []
    counter = 0
    for indices in groups.values():
        if len(indices) < 2:
            continue

        if counter > 5:
            break

        counter =+ 1
        fig, ax = plt.subplots(figsize=(8, 6))
        for idx in indices:
            resistances = df.loc[idx, distance_cols].values.astype(float) * unit_scale
            label = f'{first_col}={df.loc[idx, first_col]:.1f}'
            ax.plot(distances_um, resistances, '-o', linewidth=2, markersize=4, label=label)

        fixed_label = ', '.join(f'{c}~{df.loc[indices[0], c]:.1f}' for c in match_cols)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('Distance (μm)', fontsize=12)
        ax.set_ylabel('Resistance (kΩ)', fontsize=12)
        ax.set_title(f'Matched rows (within {rel_tolerance:.0%}): {fixed_label}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, which='both', alpha=0.3, linestyle='--')
        ax.set_xlim(33, 600)
        plt.tight_layout()
        figures.append((fig, ax))

    return figures

file_name = os.path.join(os.path.abspath('../..'), 'data_processing', 'analysis_stage', 'full_training_set_33um.csv')
figures = plot_rows_matching_all_but_first_resistivity(file_name)
plt.show()